In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q timm kagglehub opencv-python-headless matplotlib seaborn scikit-learn pillow tqdm

In [ ]:
import os
import sys
import glob
import random
import warnings
import numpy as np
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as transforms
from torchvision.utils import make_grid

import timm
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    f1_score,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
plt.style.use("dark_background")

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(" Imports complete!")

CELL 3: Download Dataset

In [ ]:
import kagglehub

print(" Downloading FaceForensics++ dataset (may take a few minutes on first run)...")
DATASET_PATH = kagglehub.dataset_download("debajyatidey/faceforensics-videos-cropped-faces")
print(f" Dataset path: {DATASET_PATH}")

# ── Discover directory structure ──
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent} {os.path.basename(root)}/  ({len(files)} files)")
    if level >= 2:
        continue

CELL 4: Dataset & DataLoaders

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

class DeepfakeDataset(Dataset):
    """Custom dataset for Real vs Fake face images."""

    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            # Return a black image on error
            image = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, label


def discover_class_folders(base_dir):

    class_paths = {"real": [], "fake": []}

    for class_name in ["real", "fake"]:
        # Try common locations
        candidates = [
            os.path.join(base_dir, class_name),
            os.path.join(base_dir, "dataset", class_name),
            os.path.join(base_dir, "faces", class_name),
        ]
        found = False
        for candidate in candidates:
            if os.path.isdir(candidate):
                img_files = sorted(
                    glob.glob(os.path.join(candidate, "**", "*.*"), recursive=True)
                )
                # Filter to image files only
                img_files = [
                    f for f in img_files
                    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
                ]
                class_paths[class_name] = img_files
                print(f"   Found {class_name}/: {len(img_files)} images")
                found = True
                break

        if not found:
            # Recursive search
            for root, dirs, _ in os.walk(base_dir):
                if os.path.basename(root).lower() == class_name:
                    img_files = sorted(
                        glob.glob(os.path.join(root, "**", "*.*"), recursive=True)
                    )
                    img_files = [
                        f for f in img_files
                        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
                    ]
                    class_paths[class_name] = img_files
                    print(f"   Found {class_name}/ (recursive): {len(img_files)} images")
                    found = True
                    break

        if not found:
            print(f"  Could not find '{class_name}' folder!")

    return class_paths


def split_dataset(class_paths, train_ratio=0.70, val_ratio=0.15, seed=42):

    rng = random.Random(seed)

    all_splits = {"train": ([], []), "val": ([], []), "test": ([], [])}

    for class_name, label in [("real", 0), ("fake", 1)]:
        imgs = class_paths[class_name][:]
        rng.shuffle(imgs)
        n = len(imgs)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_imgs = imgs[:n_train]
        val_imgs = imgs[n_train : n_train + n_val]
        test_imgs = imgs[n_train + n_val :]

        all_splits["train"][0].extend(train_imgs)
        all_splits["train"][1].extend([label] * len(train_imgs))
        all_splits["val"][0].extend(val_imgs)
        all_splits["val"][1].extend([label] * len(val_imgs))
        all_splits["test"][0].extend(test_imgs)
        all_splits["test"][1].extend([label] * len(test_imgs))

        print(f"   {class_name}: {len(train_imgs)} train / {len(val_imgs)} val / {len(test_imgs)} test")

    return all_splits["train"], all_splits["val"], all_splits["test"]


# ── Data Augmentation ──
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    transforms.RandomErasing(p=0.1),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# ── Discover & Split ──
print("\n Discovering dataset structure...")
class_paths = discover_class_folders(DATASET_PATH)

print("\n Splitting into train/val/test (70/15/15)...")
(train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels) = \
    split_dataset(class_paths, TRAIN_RATIO, VAL_RATIO, seed=SEED)

# ── Create Datasets ──
train_dataset = DeepfakeDataset(train_paths, train_labels, train_transform)
val_dataset = DeepfakeDataset(val_paths, val_labels, eval_transform)
test_dataset = DeepfakeDataset(test_paths, test_labels, eval_transform)

# ── Weighted Sampler for class balance ──
class_counts = Counter(train_labels)
class_weights = {c: 1.0 / count for c, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

# ── DataLoaders ──
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"\n Dataset Summary:")
print(f"   Train : {len(train_dataset):,} images")
print(f"   Valid : {len(val_dataset):,} images")
print(f"   Test  : {len(test_dataset):,} images")
print(f"   Batch : {BATCH_SIZE}")
print(" DataLoaders ready!")

CELL 5: Visualize Sample Images

In [ ]:
def denormalize(tensor):
    """Undo normalization for visualization."""
    return tensor * 0.5 + 0.5

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle("Sample Images from Training Set", fontsize=16, fontweight="bold", color="#00d4ff")

real_shown, fake_shown = 0, 0
for img, label in train_dataset:
    if label == 0 and real_shown < 8:
        ax = axes[0, real_shown]
        ax.imshow(denormalize(img).permute(1, 2, 0).clamp(0, 1))
        ax.set_title("REAL", color="#00ff88", fontsize=10, fontweight="bold")
        ax.axis("off")
        real_shown += 1
    elif label == 1 and fake_shown < 8:
        ax = axes[1, fake_shown]
        ax.imshow(denormalize(img).permute(1, 2, 0).clamp(0, 1))
        ax.set_title("FAKE", color="#ff4444", fontsize=10, fontweight="bold")
        ax.axis("off")
        fake_shown += 1
    if real_shown >= 8 and fake_shown >= 8:
        break

plt.tight_layout()
plt.show()

CELL 6: Build the XceptionNet Model

In [ ]:
class DeepfakeDetector(nn.Module):

    def __init__(self, pretrained=True, dropout=0.5):
        super().__init__()

        # Load pretrained XceptionNet backbone
        self.backbone = timm.create_model(
            "xception",
            pretrained=pretrained,
            num_classes=0,  # Remove original classifier
        )
        backbone_features = self.backbone.num_features  # Usually 2048

        # Custom binary classifier head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(backbone_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.6),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.4),
            nn.Linear(128, 1),
        )

        # Initialize classifier weights
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        features = self.backbone.forward_features(x)
        logits = self.classifier(features)
        return logits

    def get_features(self, x):
        """Extract feature maps for Grad-CAM."""
        return self.backbone.forward_features(x)


# ── Instantiate Model ──
model = DeepfakeDetector(pretrained=True, dropout=0.5).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Model: XceptionNet-based Deepfake Detector")
print(f"   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")
print(f"   Device          : {DEVICE}")
print(" Model ready!")

CELL 7: Training Setup

In [ ]:
# ── Hyperparameters ──
EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 3

# ── Loss, Optimizer, Scheduler ──
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=3, T_mult=2)
scaler = GradScaler()   # Mixed precision

# ── Training History ──
history = {
    "train_loss": [], "train_acc": [], "train_f1": [],
    "val_loss": [], "val_acc": [], "val_f1": [], "val_auc": [],
    "lr": [],
}


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """Train for one epoch with mixed precision."""
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc="Training", leave=False, colour="cyan")
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=(device.type == 'cuda')):
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).long().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy().astype(int))

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="binary")
    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in tqdm(loader, desc="Validating", leave=False, colour="green"):
        images = images.to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)

        with autocast(enabled=(device.type == 'cuda')):
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(outputs).cpu().numpy()
        preds = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy().astype(int))

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="binary")
    epoch_auc = roc_auc_score(all_labels, all_probs)
    return epoch_loss, epoch_acc, epoch_f1, epoch_auc

print(" Training setup complete!")
print(f"   Epochs          : {EPOCHS}")
print(f"   Learning Rate   : {LEARNING_RATE}")
print(f"   Early Stopping  : patience={PATIENCE}")

CELL 8: Train the Model

In [ ]:
print("=" * 60)
print(" TRAINING STARTED")
print("=" * 60)

best_val_auc = 0.0
best_epoch = 0
patience_counter = 0
SAVE_PATH = "best_deepfake_detector.pth"

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'─' * 60}")
    print(f" Epoch {epoch}/{EPOCHS}  |  LR: {optimizer.param_groups[0]['lr']:.2e}")
    print(f"{'─' * 60}")

    # Train
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, DEVICE
    )

    # Validate
    val_loss, val_acc, val_f1, val_auc = validate(
        model, val_loader, criterion, DEVICE
    )

    # Step scheduler
    scheduler.step()

    # Record history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)
    history["val_auc"].append(val_auc)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    # Print metrics
    print(f"   Train → Loss: {train_loss:.4f}  Acc: {train_acc:.4f}  F1: {train_f1:.4f}")
    print(f"   Valid → Loss: {val_loss:.4f}  Acc: {val_acc:.4f}  F1: {val_f1:.4f}  AUC: {val_auc:.4f}")

    # Save best model
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_auc": val_auc,
            "val_acc": val_acc,
        }, SAVE_PATH)
        print(f"    Saved best model (AUC: {val_auc:.4f})")
    else:
        patience_counter += 1
        print(f"    No improvement ({patience_counter}/{PATIENCE})")

    if patience_counter >= PATIENCE:
        print(f"\n Early stopping at epoch {epoch}!")
        break

print(f"\n{'=' * 60}")
print(f" TRAINING COMPLETE — Best AUC: {best_val_auc:.4f} (Epoch {best_epoch})")
print(f"{'=' * 60}")

CELL 9: Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Training & Validation Metrics", fontsize=18, fontweight="bold", color="#00d4ff")

epochs_range = range(1, len(history["train_loss"]) + 1)

# Loss
ax = axes[0, 0]
ax.plot(epochs_range, history["train_loss"], "o-", color="#ff6b6b", label="Train Loss", linewidth=2)
ax.plot(epochs_range, history["val_loss"], "s-", color="#4ecdc4", label="Val Loss", linewidth=2)
ax.set_title("Loss", fontsize=14, color="white")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(alpha=0.3)

# Accuracy
ax = axes[0, 1]
ax.plot(epochs_range, history["train_acc"], "o-", color="#ff6b6b", label="Train Acc", linewidth=2)
ax.plot(epochs_range, history["val_acc"], "s-", color="#4ecdc4", label="Val Acc", linewidth=2)
ax.set_title("Accuracy", fontsize=14, color="white")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(alpha=0.3)

# F1 Score
ax = axes[1, 0]
ax.plot(epochs_range, history["train_f1"], "o-", color="#ff6b6b", label="Train F1", linewidth=2)
ax.plot(epochs_range, history["val_f1"], "s-", color="#4ecdc4", label="Val F1", linewidth=2)
ax.set_title("F1 Score", fontsize=14, color="white")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(alpha=0.3)

# AUC + LR
ax = axes[1, 1]
ax.plot(epochs_range, history["val_auc"], "s-", color="#ffd93d", label="Val AUC", linewidth=2)
ax.set_title("Validation AUC-ROC", fontsize=14, color="white")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

 CELL 10: Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f" Loaded best model from epoch {checkpoint['epoch']} (AUC: {checkpoint['val_auc']:.4f})")

# Evaluate on test set
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing", colour="magenta"):
        images = images.to(DEVICE, non_blocking=True)
        with autocast(enabled=(DEVICE.type == 'cuda')):
            outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs).cpu().numpy()
        preds = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# ── Classification Report ──
print("\n" + "=" * 60)
print(" TEST SET RESULTS")
print("=" * 60)
print(classification_report(
    all_labels, all_preds,
    target_names=["REAL", "FAKE"],
    digits=4,
))

test_auc = roc_auc_score(all_labels, all_probs)
test_acc = accuracy_score(all_labels, all_preds)
print(f"   Test Accuracy : {test_acc:.4f}")
print(f"   Test AUC-ROC  : {test_auc:.4f}")

# ── Confusion Matrix ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Test Set Evaluation", fontsize=16, fontweight="bold", color="#00d4ff")

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["REAL", "FAKE"], yticklabels=["REAL", "FAKE"],
    ax=axes[0], annot_kws={"size": 14},
)
axes[0].set_title("Confusion Matrix", fontsize=14, color="white")
axes[0].set_ylabel("True Label")
axes[0].set_xlabel("Predicted Label")

# ── ROC Curve ──
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="#ffd93d", linewidth=2, label=f"AUC = {test_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[1].fill_between(fpr, tpr, alpha=0.15, color="#ffd93d")
axes[1].set_title("ROC Curve", fontsize=14, color="white")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(fontsize=12, loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

CELL 11: Grad-CAM Explainability

In [ ]:
class GradCAM:

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_backward_hook(self._backward_hook)

    def _forward_hook(self, module, input, output):
        self.activations = output.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)

        if target_class is None:
            target_class = (torch.sigmoid(output) > 0.5).long().item()

        self.model.zero_grad()
        output.backward()

        gradients = self.gradients
        activations = self.activations

        weights = gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()

        return cam.squeeze().cpu().numpy()


def visualize_gradcam(model, dataset, num_samples=8, device=DEVICE):

    # Get the last convolutional layer of XceptionNet
    target_layer = model.backbone.act4
    gradcam = GradCAM(model, target_layer)

    fig, axes = plt.subplots(3, num_samples, figsize=(num_samples * 2.5, 8))
    fig.suptitle("Grad-CAM: Where the Model Looks", fontsize=16, fontweight="bold", color="#00d4ff")

    indices = random.sample(range(len(dataset)), num_samples)

    for i, idx in enumerate(indices):
        img_tensor, label = dataset[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device).requires_grad_(True)

        # Generate Grad-CAM
        cam = gradcam.generate(input_tensor)

        # Get prediction
        with torch.no_grad():
            pred_prob = torch.sigmoid(model(img_tensor.unsqueeze(0).to(device))).item()
        pred_label = "FAKE" if pred_prob > 0.5 else "REAL"
        true_label = "FAKE" if label == 1 else "REAL"

        # Original image
        img_np = denormalize(img_tensor).permute(1, 2, 0).numpy().clip(0, 1)
        axes[0, i].imshow(img_np)
        axes[0, i].set_title(f"True: {true_label}", fontsize=9,
                             color="#00ff88" if true_label == "REAL" else "#ff4444")
        axes[0, i].axis("off")

        # Heatmap
        cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        axes[1, i].imshow(cam_resized, cmap="jet")
        axes[1, i].set_title("Grad-CAM", fontsize=9, color="white")
        axes[1, i].axis("off")

        # Overlay
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
        overlay = 0.5 * img_np + 0.5 * heatmap
        axes[2, i].imshow(overlay.clip(0, 1))
        axes[2, i].set_title(f"Pred: {pred_label} ({pred_prob:.2f})", fontsize=9,
                             color="#00ff88" if pred_label == "REAL" else "#ff4444")
        axes[2, i].axis("off")

    axes[0, 0].set_ylabel("Original", fontsize=12, color="white")
    axes[1, 0].set_ylabel("Heatmap", fontsize=12, color="white")
    axes[2, 0].set_ylabel("Overlay", fontsize=12, color="white")

    plt.tight_layout()
    plt.show()


visualize_gradcam(model, test_dataset)

CELL 12: Predict on a Single Image

In [ ]:
def predict_image(image_path, model, transform, device=DEVICE, show=True):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        with autocast(enabled=(device.type == 'cuda')):
            output = model(input_tensor).squeeze()
        prob = torch.sigmoid(output).item()

    label = "FAKE" if prob > 0.5 else "REAL"
    confidence = prob if prob > 0.5 else 1 - prob

    result = {
        "prediction": label,
        "confidence": confidence,
        "fake_probability": prob,
    }

    if show:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(image)
        color = "#ff4444" if label == "FAKE" else "#00ff88"
        ax.set_title(
            f"Prediction: {label}  |  Confidence: {confidence:.1%}",
            fontsize=14, fontweight="bold", color=color,
        )
        ax.axis("off")

        # Add confidence bar
        fig.subplots_adjust(bottom=0.15)
        bar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03])
        bar_ax.barh(0, prob, color="#ff4444", height=0.5)
        bar_ax.barh(0, 1 - prob, left=prob, color="#00ff88", height=0.5)
        bar_ax.set_xlim(0, 1)
        bar_ax.set_yticks([])
        bar_ax.set_xticks([0, 0.5, 1])
        bar_ax.set_xticklabels(["REAL", "Threshold", "FAKE"], fontsize=10)
        bar_ax.axvline(x=0.5, color="white", linewidth=2, linestyle="--")
        plt.show()

    return result


# ── Example: Predict on a test image ──
sample_path = test_paths[random.randint(0, len(test_paths) - 1)]
result = predict_image(sample_path, model, eval_transform)
print(f"\n Image: {os.path.basename(sample_path)}")
print(f"   Prediction      : {result['prediction']}")
print(f"   Confidence       : {result['confidence']:.1%}")
print(f"   Fake Probability : {result['fake_probability']:.4f}")

CELL 13: Predict on a Video

In [ ]:
def predict_video(video_path, model, transform, device=DEVICE,
                  max_frames=30, frame_interval=5, show=True):
    model.eval()
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f" Cannot open video: {video_path}")
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f" Video: {os.path.basename(video_path)}")
    print(f"   Frames: {total_frames}  |  FPS: {fps:.1f}  |  Duration: {total_frames/fps:.1f}s")

    frame_results = []
    analyzed_frames = []
    frame_idx = 0
    analyzed_count = 0

    while cap.isOpened() and analyzed_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)

            input_tensor = transform(pil_image).unsqueeze(0).to(device)
            with torch.no_grad():
                with autocast(enabled=(device.type == 'cuda')):
                    output = model(input_tensor).squeeze()
                prob = torch.sigmoid(output).item()

            frame_results.append({
                "frame": frame_idx,
                "fake_probability": prob,
                "prediction": "FAKE" if prob > 0.5 else "REAL",
            })
            analyzed_frames.append(frame_rgb)
            analyzed_count += 1

        frame_idx += 1

    cap.release()

    if not frame_results:
        print(" No frames could be analyzed.")
        return None

    # Aggregate predictions (majority voting + average probability)
    fake_probs = [r["fake_probability"] for r in frame_results]
    avg_prob = np.mean(fake_probs)
    fake_votes = sum(1 for r in frame_results if r["prediction"] == "FAKE")
    real_votes = len(frame_results) - fake_votes

    overall_pred = "FAKE" if avg_prob > 0.5 else "REAL"
    overall_conf = avg_prob if avg_prob > 0.5 else 1 - avg_prob

    result = {
        "prediction": overall_pred,
        "confidence": overall_conf,
        "avg_fake_probability": avg_prob,
        "fake_votes": fake_votes,
        "real_votes": real_votes,
        "frames_analyzed": len(frame_results),
        "frame_results": frame_results,
    }

    print(f"\n RESULT: {overall_pred} (Confidence: {overall_conf:.1%})")
    print(f"   Votes: FAKE={fake_votes}, REAL={real_votes}")
    print(f"   Avg Fake Probability: {avg_prob:.4f}")

    if show and analyzed_frames:
        # Show timeline plot
        fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [1, 3]})
        fig.suptitle(
            f"Video Analysis: {overall_pred} ({overall_conf:.1%} confidence)",
            fontsize=16, fontweight="bold",
            color="#ff4444" if overall_pred == "FAKE" else "#00ff88",
        )

        # Frame thumbnails
        n_show = min(10, len(analyzed_frames))
        thumb_indices = np.linspace(0, len(analyzed_frames) - 1, n_show, dtype=int)
        grid_img = np.concatenate(
            [cv2.resize(analyzed_frames[i], (100, 100)) for i in thumb_indices],
            axis=1,
        )
        axes[0].imshow(grid_img)
        axes[0].set_title("Sampled Frames", fontsize=12, color="white")
        axes[0].axis("off")

        # Probability timeline
        frame_nums = [r["frame"] for r in frame_results]
        axes[1].plot(frame_nums, fake_probs, "-o", color="#ffd93d", linewidth=2, markersize=4)
        axes[1].axhline(y=0.5, color="white", linewidth=1.5, linestyle="--", alpha=0.7)
        axes[1].fill_between(frame_nums, fake_probs, 0.5,
                             where=[p > 0.5 for p in fake_probs],
                             color="#ff4444", alpha=0.2, label="FAKE region")
        axes[1].fill_between(frame_nums, fake_probs, 0.5,
                             where=[p <= 0.5 for p in fake_probs],
                             color="#00ff88", alpha=0.2, label="REAL region")
        axes[1].set_xlabel("Frame Number", fontsize=12)
        axes[1].set_ylabel("Fake Probability", fontsize=12)
        axes[1].set_ylim(-0.05, 1.05)
        axes[1].legend(fontsize=10, loc="upper right")
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

    return result

CELL 14: Upload & Predict

In [ ]:
from google.colab import files as colab_files

print("=" * 60)
print(" UPLOAD AN IMAGE/VIDEO FOR DEEPFAKE DETECTION")
print("=" * 60)
print("Supported formats:")
print("   Images: .jpg, .jpeg, .png, .bmp, .webp")
print("   Videos: .mp4, .avi, .mov, .mkv, .webm")
print()

uploaded = colab_files.upload()

for filename, data in uploaded.items():
    filepath = os.path.join("/content", filename)
    with open(filepath, "wb") as f:
        f.write(data)

    ext = os.path.splitext(filename)[1].lower()

    if ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        print(f"\n  Processing image: {filename}")
        result = predict_image(filepath, model, eval_transform)
    elif ext in [".mp4", ".avi", ".mov", ".mkv", ".webm"]:
        print(f"\n Processing video: {filename}")
        result = predict_video(filepath, model, eval_transform)
    else:
        print(f"  Unsupported format: {ext}")
        continue

    print(f"\n{'─' * 40}")
    print(f"   File       : {filename}")
    print(f"   Prediction : {result['prediction']}")
    print(f"   Confidence : {result['confidence']:.1%}")
    print(f"{'─' * 40}")

CELL 15: Save / Download Model

In [ ]:
final_save = {
    "model_state_dict": model.state_dict(),
    "model_config": {
        "architecture": "XceptionNet",
        "input_size": IMG_SIZE,
        "num_classes": 1,
        "dropout": 0.5,
    },
    "training_config": {
        "dataset": "FaceForensics++ Cropped Faces (debajyatidey/faceforensics-videos-cropped-faces)",
        "epochs_trained": len(history["train_loss"]),
        "best_val_auc": best_val_auc,
        "best_epoch": best_epoch,
    },
    "history": history,
    "class_mapping": {0: "REAL", 1: "FAKE"},
}

FINAL_PATH = "deepfake_detector_final.pth"
torch.save(final_save, FINAL_PATH)
print(f" Model saved to: {FINAL_PATH}")
print(f"   File size: {os.path.getsize(FINAL_PATH) / 1e6:.1f} MB")

# Download the model (Colab only)
try:
    from google.colab import files as colab_files
    colab_files.download(FINAL_PATH)
    print(" Download started!")
except ImportError:
    print("   (Run in Google Colab to auto-download)")

print("\n" + "=" * 60)
print(" DEEPFAKE DETECTION SYSTEM COMPLETE!")
print("=" * 60)
print("""
Summary:
  --- Model trained on FaceForensics++ Cropped Faces dataset
  --- XceptionNet backbone with custom classifier
  --- Supports image and video inference
  --- Grad-CAM explainability
  --- Model saved and ready for deployment
""")